In [0]:
import requests
feed_url = dbutils.widgets.get("feed_url")
raw_path = dbutils.widgets.get("raw_path")

with requests.get(feed_url, stream=True, timeout=10) as r:
    r.raise_for_status()
    with open(raw_path, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

In [0]:
%pip install --no-deps nyct-gtfs==2.1.0

In [0]:
from nyct_gtfs.compiled_gtfs import gtfs_realtime_pb2, nyct_subway_pb2

with open(raw_path, 'rb') as f:
    binary_data = f.read() 
    message = gtfs_realtime_pb2.FeedMessage()
    message.ParseFromString(binary_data)
    print(message.header)
    print(message.entity[0])

In [0]:
from google.protobuf.json_format import MessageToJson
from pyspark.sql.functions import schema_of_json, lit

feed_json = MessageToJson(
    message,
    preserving_proto_field_name=True
)

json_df = spark.createDataFrame([feed_json], "string").toDF("json")

schema_text = json_df.select(
    schema_of_json(lit(feed_json))
).first()[0]

spark.createDataFrame([], schema_text).printSchema()

In [0]:
import json
from pyspark.sql.functions import from_json, to_json

feed_df = json_df.select(
    from_json("json", schema_text).alias("snapshot")
)

spark_json = feed_df.select(
    to_json("snapshot",
            options={"ignoreNullFields":"true"}
        ).alias("json")
).first()["json"]

orginial_data = json.loads(feed_json)
converted_data = json.loads(spark_json)

print("Data Matches:", orginial_data == converted_data)

In [0]:
display(feed_df)

In [0]:
from pathlib import Path

color_feed = Path(raw_path).stem


(
    feed_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"nyc_subway_data.default.bronze_{color_feed}")
)